# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.2 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = "task136"
CH = 10
H = W = 30
TASK_JSON = Path(COMPETITION)/"task136.json"
OUT_DIR = Path.cwd()/"task136_canvasmask_30x30"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / "task136_canvasmask_30x30_validation_summary.json"

with TASK_JSON.open("r") as f:
    task = json.load(f)

len(task["train"]), len(task["test"]), len(task["arc-gen"])


(3, 1, 262)

In [6]:
def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert ARC grid to [1,10,30,30].
    Inside real grid: one-hot.
    Outside real grid: all-zero across all channels.
    """
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def python_rule(grid):
    arr = np.array(grid, dtype=np.int64)
    out = arr.copy()
    h, w = arr.shape

    # color 1: from top-left corner, go NW
    p1 = np.argwhere(arr == 1)
    rmin1, cmin1 = p1.min(axis=0)
    r, c = int(rmin1) - 1, int(cmin1) - 1
    while r >= 0 and c >= 0:
        out[r, c] = 1
        r -= 1
        c -= 1

    # color 2: from bottom-right corner, go SE
    p2 = np.argwhere(arr == 2)
    rmax2, cmax2 = p2.max(axis=0)
    r, c = int(rmax2) + 1, int(cmax2) + 1
    while r < h and c < w:
        out[r, c] = 2
        r += 1
        c += 1

    return out.tolist()

for split in ["train", "test", "arc-gen"]:
    ok = sum(python_rule(ex["input"]) == ex["output"] for ex in task[split])
    print(split, ok, "/", len(task[split]))


train 3 / 3
test 1 / 1
arc-gen 262 / 262


In [7]:
class Task136Model(nn.Module):
    def __init__(self, h=H, w=W):
        super().__init__()
        rr = torch.arange(h, dtype=torch.float32).view(1, 1, h, 1).expand(1, 1, h, w)
        cc = torch.arange(w, dtype=torch.float32).view(1, 1, 1, w).expand(1, 1, h, w)
        self.register_buffer("R", rr)
        self.register_buffer("C", cc)

    def forward(self, x):
        # Active canvas: real ARC cells have one active channel; padding has all channels zero.
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()

        m1 = x[:, 1:2, :, :]
        m2 = x[:, 2:3, :, :]
        eps = torch.tensor(1e-6, dtype=torch.float32, device=x.device)

        # Each color block has exactly 4 pixels. Means are corner+0.5 for a 2x2 block.
        cnt1 = m1.sum(dim=(2, 3), keepdim=True) + eps
        cnt2 = m2.sum(dim=(2, 3), keepdim=True) + eps

        r1 = (m1 * self.R).sum(dim=(2, 3), keepdim=True) / cnt1
        c1 = (m1 * self.C).sum(dim=(2, 3), keepdim=True) / cnt1
        r2 = (m2 * self.R).sum(dim=(2, 3), keepdim=True) / cnt2
        c2 = (m2 * self.C).sum(dim=(2, 3), keepdim=True) / cnt2

        # Color 1 ray: same NW-SE diagonal as top-left corner, restricted to NW side.
        diag1 = ((self.R - self.C - (r1 - c1)).abs() < 0.25).float()
        northwest = ((self.R < r1) & (self.C < c1)).float()
        ray1 = diag1 * northwest * active

        # Color 2 ray: same NW-SE diagonal as bottom-right corner, restricted to SE side.
        diag2 = ((self.R - self.C - (r2 - c2)).abs() < 0.25).float()
        southeast = ((self.R > r2) & (self.C > c2)).float()
        ray2 = diag2 * southeast * active

        ch1 = ((m1 + ray1) > 0.5).float() * active
        ch2 = ((m2 + ray2) > 0.5).float() * active
        occupied = ((ch1 + ch2) > 0.5).float()
        ch0 = active * (1.0 - occupied)

        z = torch.zeros_like(ch0)
        return torch.cat([ch0, ch1, ch2, z, z, z, z, z, z, z], dim=1)

model = Task136Model().eval()


In [8]:
dummy = torch.from_numpy(grid_to_tensor_zero_padded(task["test"][0]["input"]))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

# Save shape-inferred model so intermediate value_info tensors are statically described.
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

ONNX_PATH, ONNX_PATH.stat().st_size


/tmp/ipykernel_16/2362384443.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_16/1984284707.py:15: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  eps = torch.tensor(1e-6, dtype=torch.float32, device=x.device)


(PosixPath('/kaggle/working/task136_canvasmask_30x30/task136.onnx'), 23520)

In [9]:
def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
empty_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == "" for inp in node.input)
]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print("input shape:", vi_shape(onnx_model.graph.input[0]))
print("output shape:", vi_shape(onnx_model.graph.output[0]))
print("ONNX size:", ONNX_PATH.stat().st_size)
print("ops:", dict(ops))
print("forbidden ops:", sorted(forbidden & set(ops)))
print("empty optional inputs:", len(empty_inputs))
print("non-static tensor shapes:", len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert not bad_shapes
assert ONNX_PATH.stat().st_size < 1_440_000


input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 23520
ops: {'Constant': 20, 'ReduceSum': 7, 'Greater': 6, 'Cast': 8, 'Slice': 2, 'Add': 5, 'Mul': 11, 'Div': 4, 'Sub': 5, 'Abs': 2, 'Less': 4, 'And': 2, 'Concat': 1}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [10]:
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])

def validate_split(split):
    tensor_ok = 0
    grid_ok = 0
    outside_zero_ok = 0
    bad = []
    for i, ex in enumerate(task[split]):
        x = grid_to_tensor_zero_padded(ex["input"])
        y = sess.run(None, {"input": x})[0]
        exp = grid_to_tensor_zero_padded(ex["output"])
        pred_bin = (y > 0.5).astype(np.float32)

        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)

        # Argmax grid inside true 10x10 canvas, useful for human inspection.
        h, w = len(ex["output"]), len(ex["output"][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex["output"]:
            grid_ok += 1

        active = x.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~active)) < 1e-5):
            outside_zero_ok += 1

    return {
        "tensor_exact_zero_padded": [tensor_ok, len(task[split])],
        "grid_argmax_inside_canvas": [grid_ok, len(task[split])],
        "outside_active_all_channels_zero": [outside_zero_ok, len(task[split])],
        "bad_indices": bad[:10],
    }

summary = {
    "task_id": TASK_ID,
    "rule": "color 1 extends NW from its 2x2 block top-left; color 2 extends SE from its 2x2 block bottom-right; output masked by active canvas",
    "onnx_path": str(ONNX_PATH),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "empty_optional_inputs": len(empty_inputs),
    "non_static_tensor_shapes": len(bad_shapes),
    "validation": {split: validate_split(split) for split in ["train", "test", "arc-gen"]},
}

print(json.dumps(summary, indent=2)[:4000])
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)


{
  "task_id": "task136",
  "rule": "color 1 extends NW from its 2x2 block top-left; color 2 extends SE from its 2x2 block bottom-right; output masked by active canvas",
  "onnx_path": "/kaggle/working/task136_canvasmask_30x30/task136.onnx",
  "onnx_size_bytes": 23520,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 20,
    "ReduceSum": 7,
    "Greater": 6,
    "Cast": 8,
    "Slice": 2,
    "Add": 5,
    "Mul": 11,
    "Div": 4,
    "Sub": 5,
    "Abs": 2,
    "Less": 4,
    "And": 2,
    "Concat": 1
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "validation": {
    "train": {
      "tensor_exact_zero_padded": [
        3,
        3
      ],
      "grid_argmax_inside_canvas": [
        3,
        3
      ],
      "outside_active_all_channels_zero": [
        3,
        3
      ],
      "bad_indices": []
    },
    "test": {
      "tensor_exact_zero_padd

In [11]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]


Wrote: /kaggle/working/submission.zip
Zip contents: ['task136.onnx']
